# Architecture Compiler — **Standalone Interactive Notebook**
This version runs **without the repository, local files, `archcompiler` installation, or API key**. It creates its own example specifications in memory. The deterministic compilation demo uses Python's standard library. Optional PyYAML provides readable YAML input/output; without it, JSON is used (JSON is a valid subset of YAML).

**Safe by design:** no Terraform apply, cloud changes, API requests, or remote execution. Generated code is illustrative and requires review.

In [ ]:
import json, re, copy
from pathlib import Path
try:
    import yaml
    YAML_AVAILABLE = True
except ImportError:
    yaml = None
    YAML_AVAILABLE = False
print('Standalone ready. YAML rendering:', 'PyYAML' if YAML_AVAILABLE else 'JSON fallback (valid YAML)')

## 1. Example canonical specifications (editable)
Change the values below, then rerun subsequent cells. Unlike the prior notebook, there are no references to `examples/` or imports from the repository.

In [ ]:
network_spec = {
    'apiVersion': 'architecture.example/v1',
    'kind': 'InfrastructureSpecification',
    'resources': {'core_network': {'type':'network', 'network_access':'private', 'cidr_block':'10.0.0.0/16'}},
    'policies': [{'id':'private-network', 'subject':'core_network', 'rule':'deny_public_access', 'enforcement':'mandatory'}],
    'targets': ['terraform', 'rego'], 'review_required': True,
}
application_spec = {
    'apiVersion': 'architecture.example/v1', 'kind': 'InfrastructureSpecification',
    'resources': {'application': {'type':'compute','replicas':3,'availability_zones':2},
                  'database': {'type':'relational_database','network_access':'private'}},
    'relationships':[{'from':'application','to':'database','type':'network_access'}],
    'policies':[{'id':'database-private-only','subject':'database','rule':'deny_public_access','enforcement':'mandatory'},
                {'id':'multi-zone','subject':'application','rule':'minimum_availability_zones','value':2,'enforcement':'mandatory'}],
    'targets':['rego'], 'review_required':True,
}
def show_yaml(value):
    return yaml.safe_dump(value,sort_keys=False) if YAML_AVAILABLE else json.dumps(value,indent=2)
print(show_yaml(application_spec))

## 2. Validate the canonical IR and enforce mandatory constraints
Only documented resource and rule types are accepted. Missing values and unsafe policies raise errors instead of being silently guessed.

In [ ]:
RESOURCE_TYPES = {'network', 'compute', 'relational_database'}
POLICY_RULES = {'deny_public_access', 'minimum_availability_zones'}
def validate_spec(spec):
    if not isinstance(spec,dict) or spec.get('kind') != 'InfrastructureSpecification':
        raise ValueError('Expected kind: InfrastructureSpecification')
    resources = spec.get('resources')
    if not isinstance(resources,dict) or not resources:
        raise ValueError('At least one resource is required')
    for name, resource in resources.items():
        if not re.fullmatch(r'[A-Za-z][A-Za-z0-9_]*', name):
            raise ValueError(f'Invalid resource ID: {name}')
        if resource.get('type') not in RESOURCE_TYPES:
            raise ValueError(f'Unsupported resource type: {name}')
        if resource.get('network_access','private') not in {'private','public'}:
            raise ValueError(f'Invalid access for {name}')
        if not isinstance(resource.get('availability_zones',1),int) or not 1 <= resource.get('availability_zones',1) <= 6:
            raise ValueError(f'Invalid availability zone count: {name}')
        if resource['type'] == 'network':
            import ipaddress
            cidr = resource.get('cidr_block')
            if not cidr: raise ValueError(f'Missing explicit CIDR for network {name}')
            ipaddress.ip_network(cidr,strict=True)
    for relation in spec.get('relationships',[]):
        if relation.get('from') not in resources or relation.get('to') not in resources:
            raise ValueError(f'Unknown relationship endpoint: {relation}')
    for policy in spec.get('policies',[]):
        if policy.get('subject') not in resources or policy.get('rule') not in POLICY_RULES:
            raise ValueError(f'Invalid policy: {policy}')
        if policy['rule'] == 'minimum_availability_zones' and (not isinstance(policy.get('value'),int) or policy['value'] < 1):
            raise ValueError('minimum_availability_zones requires positive integer value')
        if policy.get('enforcement','mandatory') not in {'mandatory','advisory'}:
            raise ValueError('Invalid enforcement level')
    return True

def check_constraints(spec):
    validate_spec(spec)
    violations=[]
    for p in spec.get('policies',[]):
        if p.get('enforcement','mandatory') != 'mandatory': continue
        r=spec['resources'][p['subject']]
        if p['rule']=='deny_public_access' and r.get('network_access','private')=='public':
            violations.append(f"{p['id']}: {p['subject']} is public")
        if p['rule']=='minimum_availability_zones' and r.get('availability_zones',1)<p['value']:
            violations.append(f"{p['id']}: {p['subject']} has fewer than {p['value']} zones")
    return violations
print('Application valid:', validate_spec(application_spec))
print('Policy violations:', check_constraints(application_spec))

## 3. Compile policies to Rego and supported networks to Terraform JSON
The Rego backend evaluates the normalized architecture specification, **not a raw Terraform plan**. The Terraform backend supports **only AWS VPCs**; compute and databases fail closed.

In [ ]:
def compile_rego(spec):
    validate_spec(spec)
    lines=['package architecture.guardrails','', 'import rego.v1', '']
    mandatory=[p for p in spec.get('policies',[]) if p.get('enforcement','mandatory')=='mandatory']
    for p in mandatory:
        subject=json.dumps(p['subject'])
        if p['rule']=='deny_public_access':
            condition=f'input.resources[{subject}].network_access == "public"'
            msg=p['id']+': public network access is forbidden'
        else:
            condition=f"input.resources[{subject}].availability_zones < {p['value']}"
            msg=p['id']+': availability zone minimum not satisfied'
        lines += ['deny contains msg if {', '  '+condition, '  msg := '+json.dumps(msg), '}', '']
    if not mandatory: lines += ['deny := set()']
    return '\n'.join(lines)+'\n'

def compile_terraform(spec):
    validate_spec(spec)
    errors=check_constraints(spec)
    if errors: raise ValueError('Policy violation: '+'; '.join(errors))
    unsupported=[name for name,r in spec['resources'].items() if r['type']!='network']
    if unsupported:
        raise ValueError('Terraform backend supports AWS network resources only: '+', '.join(unsupported))
    vpcs={name:{'cidr_block':r['cidr_block'],'enable_dns_hostnames':True,'tags':{'Name':name}}
          for name,r in spec['resources'].items()}
    output={'terraform':{'required_providers':{'aws':{'source':'hashicorp/aws','version':'~> 5.0'}}},
            'provider':{'aws':{'region':'${var.aws_region}'}},
            'variable':{'aws_region':{'type':'string','description':'Set AWS region explicitly'}},
            'resource':{'aws_vpc':vpcs},
            'output':{'architecture_review_required':{'value':True}}}
    return json.dumps(output,indent=2)+'\n'

def compile_target(spec,target):
    errors=check_constraints(spec)
    if errors: raise ValueError('Policy violation: '+'; '.join(errors))
    if target=='rego': return compile_rego(spec)
    if target=='terraform': return compile_terraform(spec)
    raise ValueError(f'Unsupported target: {target}')
print(compile_target(application_spec,'rego'))
print('Generated AWS VPC Terraform JSON:')
print(compile_target(network_spec,'terraform'))

## 4. Change a policy-relevant value and see the compiler reject it

In [ ]:
unsafe = copy.deepcopy(application_spec)
unsafe['resources']['database']['network_access'] = 'public'
print('Violations:',check_constraints(unsafe))
try:
    compile_target(unsafe,'rego')
except ValueError as exc:
    print('EXPECTED BLOCK:',exc)
else:
    raise AssertionError('Unsafe specification was unexpectedly allowed')

## 5. Simple commented-text interpretation (offline, deterministic example)
This rule-based demonstration recognizes **only** a narrow phrasing. It does not understand arbitrary diagrams or natural language. For broader interpretations use the optional LLM cell later, and review its outputs.

In [ ]:
def interpret_known_network_requirement(text):
    low=text.lower()
    if not ('network' in low and 'private' in low and '10.0.0.0/16' in low):
        raise ValueError('Ambiguous or unsupported requirements: manual review required')
    result=copy.deepcopy(network_spec)
    result['evidence']=[{'claim':'private network with CIDR 10.0.0.0/16',
                          'source':'inline requirements', 'certainty':'explicit'}]
    return result
notes = '# Provision a private AWS network with CIDR 10.0.0.0/16. Deny public access.'
interpreted=interpret_known_network_requirement(notes)
print(show_yaml(interpreted))
print('Violations:',check_constraints(interpreted))

## 6. Optional: Load or save real YAML files
Runs without PyYAML using JSON files. Use `pip install pyyaml` to enable reading/writing ordinary YAML syntax.

In [ ]:
def read_spec_file(path):
    path=Path(path)
    content=path.read_text(encoding='utf-8')
    result=yaml.safe_load(content) if YAML_AVAILABLE else json.loads(content)
    validate_spec(result)
    return result

def save_spec_file(spec,path):
    validate_spec(spec)
    Path(path).write_text(show_yaml(spec),encoding='utf-8')
    return Path(path)
# Example: to write files, uncomment the following line:
# save_spec_file(application_spec, 'application.yaml' if YAML_AVAILABLE else 'application.json')
print('YAML file support:', YAML_AVAILABLE, '| JSON files supported:',True)

## 7. Optional LLM interpretation using OpenAI
**Disabled by default.** Requires `pip install openai`, an API key, and user review. The LLM proposes JSON/YAML; the local compiler validates it. Never put credentials inside the notebook.

In [ ]:
def interpret_with_ai_optional(text, model='gpt-4o-mini'):
    import os
    if not os.getenv('OPENAI_API_KEY'):
        raise RuntimeError('Set OPENAI_API_KEY securely in your environment first')
    from openai import OpenAI
    response=OpenAI().chat.completions.create(
        model=model, response_format={'type':'json_object'}, temperature=0,
        messages=[{'role':'system','content':(
            'Return only a JSON object representing an InfrastructureSpecification. '
            'Allowed resource types: network, compute, relational_database. '
            'Allowed policies: deny_public_access, minimum_availability_zones. '
            'Do not invent details. For network resources require a source-explicit CIDR. '
            'For ambiguous requirements omit them. Always set review_required=true.')},
            {'role':'user','content':text}])
    proposed=json.loads(response.choices[0].message.content)
    validate_spec(proposed)
    proposed['review_required']=True
    return proposed
# Example (only run after configuring API):
# ai_spec = interpret_with_ai_optional(notes)
# print(show_yaml(ai_spec))
print('LLM feature available as an opt-in function; no API call made.')

## 8. Executable self-tests — run all cells to verify

In [ ]:
assert validate_spec(network_spec)
assert validate_spec(application_spec)
assert check_constraints(application_spec)==[]
assert len(check_constraints(unsafe))==1
assert 'deny contains msg' in compile_target(application_spec,'rego')
generated=json.loads(compile_target(network_spec,'terraform'))
assert generated['resource']['aws_vpc']['core_network']['cidr_block']=='10.0.0.0/16'
try:
    compile_target(application_spec,'terraform')
except ValueError as exc:
    assert 'network resources only' in str(exc)
else:
    raise AssertionError('Unsupported compute/database unexpectedly compiled')
assert validate_spec(interpreted)
print('PASS: eight standalone checks; no repository or cloud access used.')